# <center> <img src="../../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Lab 04** </center>
---
### <center> **Nicolas Navarro Valenzuela 746812** </center>
---
**Profesor**: Pablo Camarillo Ramirez

In [1]:
from spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("spark://spark-master:7077", "lab04 - Data Unions and Joins Pipeline")
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 23:03:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!pwd

/opt/spark/work-dir


In [7]:
!ls /opt/spark/work-dir/data/car_service/agencies/

agencies.csv


In [2]:
from pyspark.sql.functions import get_json_object

"agency_id, agency_info"

column_types = [
    ("agency_id", "int"),
    ("agency_info", "string")
]

agencies_schema = SparkUtils.generate_schema(column_types)
agencies_df = (su._spark
                 .read
                 .option("header", "true")
                 .schema(agencies_schema)
                 .csv("/opt/spark/work-dir/data/car_service/agencies/agencies.csv"))

agencies_df = (agencies_df.withColumn("agency name",get_json_object("agency_info", "$.agency_name"))
                        .withColumn("agency city",get_json_object("agency_info", "$.city")))

agencies_df = agencies_df.select("agency_id", "agency name", "agency city")

agencies_df.show(truncate=False)


+---------+-------------+-------------+
|agency_id|agency name  |agency city  |
+---------+-------------+-------------+
|1        |NYC Rentals  |New York     |
|2        |LA Car Rental|Londres      |
|3        |Zapopan Auto |Zapopan      |
|4        |SF Cars      |San Francisco|
|5        |Mexico Cars  |Mexico City  |
+---------+-------------+-------------+



In [3]:
"brand_id, brand_info"

column_types = [
    ("brand_id", "int"),
    ("brand_info", "string")
]

brands_schema = SparkUtils.generate_schema(column_types)
brands_df = (su._spark
                .read
                .option("header", "true")
                .schema(brands_schema)
                .csv("/opt/spark/work-dir/data/car_service/brands/brands.csv")
)

brands_df = (brands_df.withColumn("brand name",get_json_object("brand_info", "$.brand_name"))
                        .withColumn("brand country",get_json_object("brand_info", "$.country")))

brands_df = brands_df.select("brand_id", "brand name", "brand country")

brands_df.show(truncate=False)

+--------+-------------+-------------+
|brand_id|brand name   |brand country|
+--------+-------------+-------------+
|1       |Mercedes-Benz|Germany      |
|2       |BMW          |Germany      |
|3       |Audi         |Germany      |
|4       |Ford         |US           |
|5       |BYD          |China        |
|6       |Honda        |Japan        |
|7       |Toyota       |Japan        |
+--------+-------------+-------------+



In [4]:
"car_id, car_info"

column_types = [
    ("car_id", "int"),
    ("car_info", "string")
]

cars_schema = SparkUtils.generate_schema(column_types)
cars_df = (su._spark
            .read
            .option("header", "true")
            .schema(cars_schema)
            .csv("/opt/spark/work-dir/data/car_service/cars/cars.csv")
)

cars_df = (cars_df.withColumn("car name",get_json_object("car_info", "$.car_name"))
                    .withColumn("car brand id",get_json_object("car_info", "$.brand_id"))
                    .withColumn("price per day",get_json_object("car_info", "$.price_per_day")))

cars_df = cars_df.select("car_id", "car name", "car brand id", "price per day")
print(f"Total records: {cars_df.count()}")
cars_df.show(10, truncate=False)

Total records: 29
+------+------------------------------------+------------+-------------+
|car_id|car name                            |car brand id|price per day|
+------+------------------------------------+------------+-------------+
|1     |Chang-Fisher Model 7                |5           |139          |
|2     |Sheppard-Tucker Model 4             |6           |70           |
|3     |Faulkner-Howard Model 5             |3           |53           |
|4     |Wagner LLC Model 1                  |5           |89           |
|5     |Campos PLC Model 4                  |4           |112          |
|6     |Archer-Patel Model 2                |3           |55           |
|7     |Patrick, Barrera and Collins Model 6|6           |66           |
|8     |Jones, Jefferson and Rivera Model 7 |2           |115          |
|9     |Garcia, Hamilton and Carr Model 5   |6           |137          |
|10    |Levy Group Model 9                  |3           |93           |
+------+-------------------------

In [5]:
"customer_id, customer_info"

column_types = [
    ("customer_id", "int"),
    ("customer_info", "string")
]

customers_schema = SparkUtils.generate_schema(column_types)
customers_df = (su._spark
                    .read
                    .option("header", "true")
                    .schema(customers_schema)
                    .csv("/opt/spark/work-dir/data/car_service/customers/customers.csv")
)

customers_df = (customers_df.withColumn("customer name",get_json_object("customer_info", "$.customer_name"))
                            .withColumn("customer city",get_json_object("customer_info", "$.city"))
                            .withColumn("customer age",get_json_object("customer_info", "$.age")))

customers_df = customers_df.select("customer_id", "customer name", "customer city", "customer age")
print(f"Total records: {customers_df.count()}")
customers_df.show(10, truncate=False)

Total records: 159
+-----------+----------------+-------------+------------+
|customer_id|customer name   |customer city|customer age|
+-----------+----------------+-------------+------------+
|1          |Tiffany Riley   |Monterrey    |32          |
|2          |Matthew Davies  |Monterrey    |36          |
|3          |Rebecca Miller  |Mexico City  |30          |
|4          |Katelyn Mccoy   |New York     |34          |
|5          |Dana Dennis     |Zapopan      |26          |
|6          |Daniel Norton   |Mexico City  |34          |
|7          |Robert Garcia   |Zapopan      |47          |
|8          |Michael Williams|Monterrey    |33          |
|9          |Susan Ferguson  |San Francisco|41          |
|10         |Alyssa Reid     |Mexico City  |40          |
+-----------+----------------+-------------+------------+
only showing top 10 rows


In [6]:
"rental_id,rental_info"

column_types = [
    ("rental_id", "int"),
    ("rental_info", "string")
]
rentals_schema = SparkUtils.generate_schema(column_types)
rentals_df = (su._spark
                .read
                .option("header", "true")
                .schema(rentals_schema)
                .csv("/opt/spark/work-dir/data/car_service/rentals/")
)

rentals_df = (rentals_df.withColumn("car id",get_json_object("rental_info", "$.car_id"))
                        .withColumn("customer id",get_json_object("rental_info", "$.customer_id"))
                        .withColumn("agency id",get_json_object("rental_info", "$.agency_id"))
)

rentals_df = rentals_df.select("rental_id", "car id", "customer id", "agency id")
print(f"Total records: {rentals_df.count()}")
rentals_df.show(10, truncate=False)


Total records: 17834
+---------+------+-----------+---------+
|rental_id|car id|customer id|agency id|
+---------+------+-----------+---------+
|11891    |21    |71         |1        |
|11892    |11    |52         |2        |
|11893    |22    |116        |4        |
|11894    |5     |107        |1        |
|11895    |4     |53         |4        |
|11896    |8     |131        |2        |
|11897    |23    |66         |3        |
|11898    |24    |60         |4        |
|11899    |27    |92         |2        |
|11900    |3     |40         |4        |
+---------+------+-----------+---------+
only showing top 10 rows


In [7]:
#Final dataframe with all the information

final_df = rentals_df.join(agencies_df, rentals_df["agency id"] == agencies_df["agency_id"], "inner") \
                    .join(cars_df, rentals_df["car id"] == cars_df["car_id"], "inner") \
                    .join(brands_df, cars_df["car brand id"] == brands_df["brand_id"], "inner") \
                    .join(customers_df, rentals_df["customer id"] == customers_df["customer_id"], "inner")

final_df = final_df.select("rental_id", "car name", "agency name", "customer name") #.filter(final_df["customer name"] == "Margaret Jones")
print(f"Total Records: {final_df.count()}")
final_df.show(5, truncate=False)


Total Records: 17834
+---------+-----------------------+-------------+---------------+
|rental_id|car name               |agency name  |customer name  |
+---------+-----------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9|NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8   |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5  |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4     |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1     |SF Cars      |Kristin Potts  |
+---------+-----------------------+-------------+---------------+
only showing top 5 rows


In [9]:
final_df.write \
        .partitionBy("agency name") \
        .mode("overwrite") \
        .parquet("hdfs://namenode:9000/output/agencies_car_service")

In [10]:
su._spark.stop()